# 🧬 Pipeline scRNA-seq — IBD Colon/Recto - GSE214695

## Integración Harmony + Umap y evaluación

**Flujo de trabajo:**
1. Cargar objeto del Paso 3
2. Ejecutar Harmony → X_pca_harmony
3. Calcular vecindario y UMAP sobre X_pca_harmony
4. Visualizar pre vs post integración
5. Evaluar calidad de la integración con métricas cuantitativas
6. Guardar objeto integrado

# · Importaciones y configuración

In [ ]:
# Montar Google Drive y prepara el gestor de entornos Conda
from google.colab import drive
drive.mount('/content/drive')

!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


In [ ]:
# Instalación del entorno Conda del proyecto (environment.yml), con las
# versiones exactas de todas las dependencias fijadas para reproducibilidad.

!conda env update -n base -f /content/drive/MyDrive/TFM_IBD_GSE214695/environment.yml -q

Retrieving notices: ...working... done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import scanpy as sc
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder
import harmonypy as hm
import yaml

sc.settings.verbosity = 1

# · Configuración de rutas

In [ ]:
REPO_ROOT = Path("/content/drive/MyDrive/TFM_IBD_GSE214695")
with open(REPO_ROOT / "config" / "params.yaml") as f:
    PARAMS = yaml.safe_load(f)

PROJECT_ROOT = os.environ.get(PARAMS["project_root_env_var"], PARAMS["project_root_default"])

INPUT_PATH  = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['03_diagnostics']}/{PARAMS['outputs']['diagnosed_h5ad']}"
OUTPUT_DIR  = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['04_integrated']}"
OUTPUT_PATH = f"{OUTPUT_DIR}/{PARAMS['outputs']['integrated_h5ad']}"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
FIGURES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['figures']['04_integration']}"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)

BATCH_KEY     = "sample_id"       # columna que identifica cada paciente/muestra
CONDITION_KEY = "condition"       # columna con HC / UC / CD
N_PCS         = 20                # PCs elegidos según el elbow plot
PALETTE       = {'HC': '#2E86AB', 'UC': '#E84855', 'CD': '#F4A261'}

print("Configuración cargada")
print(f"Usando {N_PCS} PCs")


Configuración cargada
Usando 20 PCs


In [ ]:
adata = sc.read_h5ad(INPUT_PATH)
print(f"\n→ {adata.n_obs:,} células × {adata.n_vars:,} genes")
assert 'X_pca' in adata.obsm, "ERROR: ejecuta Paso 2 primero"
assert BATCH_KEY in adata.obs.columns, f"ERROR: columna '{BATCH_KEY}' no encontrada"
assert CONDITION_KEY in adata.obs.columns, "ERROR: ejecuta Paso 3 para añadir condition"


→ 43,388 células × 33,538 genes


# 🔬 Ejecutar Harmony



In [ ]:
# Extraer las N_PCS componentes que queremos corregir formato: (n_cells, n_pcs)
pcs = adata.obsm['X_pca'][:, :N_PCS].copy()
print(f"   PCA input shape: {pcs.shape}")  # debe ser (43388, 20)

# Ejecutar Harmony directamente
# pcs     : matriz (n_cells × n_pcs) — NO transponer
# adata.obs: metadatos de células (contiene sample_id)
# BATCH_KEY: nombre de la columna con el identificador de batch
harmony_out = hm.run_harmony(
    pcs,
    adata.obs,
    BATCH_KEY,
    max_iter_harmony=50,
    random_state=42
)

# Z_corr ya está en formato (n_cells, n_pcs)
adata.obsm['X_pca_harmony'] = harmony_out.Z_corr
print(f"   X_pca_harmony shape: {adata.obsm['X_pca_harmony'].shape}")  # debe ser (43388, 20)

# Verificar que la corrección se aplicó realmente
diff = np.abs(pcs - adata.obsm['X_pca_harmony']).mean()
print(f"   Diferencia media pre/post: {diff:.4f}")
if diff < 0.01:
    print("Corrección muy pequeña")
else:
    print("Corrección aplicada correctamente")

2026-08-29 08:03:23,942 - harmonypy - INFO - Running Harmony
2026-08-29 08:03:23,943 - harmonypy - INFO -   Parameters:
2026-08-29 08:03:23,944 - harmonypy - INFO -     max_iter_harmony: 50
2026-08-29 08:03:23,945 - harmonypy - INFO -     max_iter_kmeans: 4
2026-08-29 08:03:23,946 - harmonypy - INFO -     epsilon_cluster: 0.001
2026-08-29 08:03:23,948 - harmonypy - INFO -     epsilon_harmony: 0.01
2026-08-29 08:03:23,949 - harmonypy - INFO -     nclust: 100
2026-08-29 08:03:23,949 - harmonypy - INFO -     block_size: 0.05
2026-08-29 08:03:23,950 - harmonypy - INFO -     lamb: dynamic (alpha=0.2)
2026-08-29 08:03:23,952 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-08-29 08:03:23,953 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-08-29 08:03:23,954 - harmonypy - INFO -     verbose: True
2026-08-29 08:03:23,956 - harmonypy - INFO -     random_state: 42
2026-08-29 08:03:23,957 - harmonypy - INFO -   Data: 20 PCs × 43388 cells
2

   PCA input shape: (43388, 20)


2026-08-29 08:03:24,823 - harmonypy - INFO - Iteration 1 of 50
2026-08-29 08:03:26,027 - harmonypy - INFO - Iteration 2 of 50
2026-08-29 08:03:27,363 - harmonypy - INFO - Iteration 3 of 50
2026-08-29 08:03:28,298 - harmonypy - INFO - Iteration 4 of 50
2026-08-29 08:03:29,165 - harmonypy - INFO - Iteration 5 of 50
2026-08-29 08:03:30,034 - harmonypy - INFO - Iteration 6 of 50
2026-08-29 08:03:30,901 - harmonypy - INFO - Iteration 7 of 50
2026-08-29 08:03:31,766 - harmonypy - INFO - Iteration 8 of 50
2026-08-29 08:03:32,631 - harmonypy - INFO - Converged after 8 iterations


   X_pca_harmony shape: (43388, 20)
   Diferencia media pre/post: 0.4311
Corrección aplicada correctamente


# · Vecindario y UMAP sobre las coordenadas corregidas por Harmony



In [ ]:
print("\n→ Calculando vecindario y UMAP post-Harmony...")

sc.pp.neighbors(adata,
                n_pcs=N_PCS,
                use_rep='X_pca_harmony',
                n_neighbors=15,
                key_added='neighbors_harmony')

sc.tl.umap(adata,
           neighbors_key='neighbors_harmony',
           min_dist=0.3,
           random_state=42)

adata.obsm['X_umap_harmony'] = adata.obsm['X_umap'].copy()

print("   UMAP post-Harmony guardado")



→ Calculando vecindario y UMAP post-Harmony...
   UMAP post-Harmony guardado


# · Visualización: comparación pre vs post Harmony


In [ ]:
print("\n→ Generando figura comparativa...")

umap_pre = adata.obsm['X_umap_uncorrected']
umap_post = adata.obsm['X_umap_harmony']

samples = adata.obs[BATCH_KEY].values
conditions = adata.obs[CONDITION_KEY].values

sample_list = sorted(adata.obs[BATCH_KEY].unique())
cmap_samples = plt.get_cmap('tab20', len(sample_list))

CONDITIONS = ['HC', 'UC', 'CD']

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle("Integración Harmony — pre vs post",
             fontsize=13,
             fontweight='bold')

titles = [
    ("Antes — por muestra", "sample", "pre"),
    ("Después — por muestra", "sample", "post"),
    ("Antes — por condición", "condition", "pre"),
    ("Después — por condición", "condition", "post")
]

for ax, (title, color_by, timing) in zip(axes.flatten(), titles):
    umap = umap_pre if timing == "pre" else umap_post
    if color_by == "sample":
        for i, s in enumerate(sample_list):
            mask = samples == s
            ax.scatter(umap[mask,0],
                       umap[mask,1],
                       s=0.8,
                       alpha=0.3,
                       color=cmap_samples(i),
                       label=s,
                       rasterized=True)
        if timing == "post":
            ax.legend(markerscale=5,
                      bbox_to_anchor=(1.01,1),
                      loc='upper left',
                      fontsize=6)
    else:
        for cond in CONDITIONS:
            mask = conditions == cond
            ax.scatter(umap[mask,0],
                       umap[mask,1],
                       s=0.8,
                       alpha=0.3,
                       color=PALETTE[cond],
                       label=cond,
                       rasterized=True)
        ax.legend(markerscale=8, fontsize=10)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/harmony_pre_vs_post_comparison.png",
            dpi=120,
            bbox_inches='tight')
plt.close()
print("   → Guardada: harmony_pre_vs_post_comparison.png")



→ Generando figura comparativa...
   → Guardada: harmony_pre_vs_post_comparison.png


# · Evaluación cuantitativa de la integración



In [ ]:
print("\n→ Evaluando integración...")

# Codificar etiquetas
le = LabelEncoder()
cond_enc = le.fit_transform(conditions)
batch_enc = le.fit_transform(samples)

# Silhouette por condición
sil_cond_pre = silhouette_score(adata.obsm['X_pca'][:,:N_PCS],
                                cond_enc,
                                sample_size=5000,
                                random_state=42)
sil_cond_post = silhouette_score(adata.obsm['X_pca_harmony'][:,:N_PCS],
                                 cond_enc,
                                 sample_size=5000,
                                 random_state=42)

# Silhouette por batch (queremos que baje)
sil_batch_pre = silhouette_score(adata.obsm['X_pca'][:,:N_PCS],
                                 batch_enc,
                                 sample_size=5000,
                                 random_state=42)
sil_batch_post = silhouette_score(adata.obsm['X_pca_harmony'][:,:N_PCS],
                                  batch_enc,
                                  sample_size=5000,
                                  random_state=42)

print(f"   Silhouette condición: pre={sil_cond_pre:.4f} → post={sil_cond_post:.4f}")
print(f"   Silhouette batch    : pre={sil_batch_pre:.4f} → post={sil_batch_post:.4f}")

# Figura resumen de métricas
fig, ax = plt.subplots(figsize=(8,5))
metrics = ['Silhouette\ncondición', 'Silhouette\nbatch']
pre_vals = [sil_cond_pre, sil_batch_pre]
post_vals = [sil_cond_post, sil_batch_post]

x = [0,1]
width = 0.3

ax.bar([i-width/2 for i in x], pre_vals, width, label='Pre-Harmony', color='#ADB5BD')
ax.bar([i+width/2 for i in x], post_vals, width, label='Post-Harmony', color='#2E86AB')
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylabel('Silhouette score')
ax.set_title('Evaluación integración Harmony')
ax.legend()

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/harmony_integration_metrics.png", dpi=120)
plt.close()

print("   → Guardada: harmony_integration_metrics.png")


→ Evaluando integración...
   Silhouette condición: pre=-0.0113 → post=-0.0208
   Silhouette batch    : pre=-0.1597 → post=-0.1675
   → Guardada: harmony_integration_metrics.png


# · Guardar objeto integrado

In [ ]:
print(f"\n→ Guardando objeto en {OUTPUT_PATH} ...")

adata.write_h5ad(OUTPUT_PATH, compression='gzip')


print("\n INTEGRACIÓN COMPLETADA")
print(f"   X_pca_harmony shape: {adata.obsm['X_pca_harmony'].shape}")
